### Update projected starting lineups

In [ ]:
from MODELS.scrapStarting import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from FEATURES.featuresV2 import *
from PRODUCTION.calculateEVS import *
from PRODUCTION.pipeline import *
from PRODUCTION.teamInfo import teamStarPlayer, projectedStartingFive, mainStartingFive

### Load Model

In [6]:
# Load split NGBoost models (mean, variance, calibration factor, and isotonic calibrator)
pts_mean_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_MEAN_MODEL_PRODUCTION.pkl')
pts_var_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_VAR_MODEL_PRODUCTION.pkl')
calibration_factor = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_CALIBRATION_FACTOR_PRODUCTION.pkl')

model = (pts_mean_model, pts_var_model, calibration_factor)  
features = joblib.load('../MODELS/SAVED_MODELS/feature_list.pkl')

print(f"Loaded models with calibration factor: {calibration_factor}")

Loaded models with calibration factor: 4.5


### Load Player Data and Bookmaker Data

In [7]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')

usData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')

dfsData.head()

/var/folders/9q/5_554qsx5z70w9d_vkmvjg0h0000gn/T/ipykernel_52525/1321450873.py:5: DtypeWarning: Columns (33) have mixed types. Specify dtype option on import or set low_memory=False.
  s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE
0,DraftKings Pick6,player_points,Donovan Mitchell,Over,27.5,-137,2025-11-22,2025-11-22T01:57:49Z
1,DraftKings Pick6,player_points,Donovan Mitchell,Under,27.5,-137,2025-11-22,2025-11-22T01:57:49Z
2,DraftKings Pick6,player_points,Jaylen Brown,Over,21.5,-137,2025-11-22,2025-11-22T01:57:50Z
3,DraftKings Pick6,player_points,Jaylen Brown,Under,21.5,-137,2025-11-22,2025-11-22T01:57:50Z
4,DraftKings Pick6,player_points,Brandon Ingram,Over,21.5,-137,2025-11-22,2025-11-22T01:52:02Z


### Top EVs for single bets

In [8]:
singlePTSBookies = usData[(usData['CATEGORY'] == 'player_points') & (usData['BOOKMAKER'] != 'Bovada') & (usData['BOOKMAKER'] != 'BetOnline.ag')]

singleBets = calculateSingleBets(s26, singlePTSBookies, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10)



singleBets = singleBets[['NAME', 'BOOKMAKER','LINE', 'PREDICTION', 'SIDE','ODDS','RECOMMENDATION', 'EV$', 'KELLY_FRACTION','SIGMA FLAG']].head(15)
singleBets.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/singleBets.csv', index=False)
singleBets.head(5)

Processing single bets...
Pre-computing predictions for 139 unique players...


,NAME,BOOKMAKER,LINE,PREDICTION,SIDE,ODDS,RECOMMENDATION,EV$,KELLY_FRACTION,SIGMA FLAG
189,Jerami Grant,DraftKings,22.5,17.10,Under,-111,1,6.09,0.676,Med
1060,Bennedict Mathurin,BetMGM,21.5,26.02,Over,110,1,5.92,0.538,High
854,Alperen Sengun,BetRivers,24.5,28.14,Over,112,0,5.09,0.455,High
444,Keyonte George,FanDuel,18.5,23.11,Over,100,1,5.01,0.501,High
1140,Tre Jones,BetMGM,9.5,13.85,Over,-110,1,4.76,0.524,Med


## Top EVs for 2 leg bets

### Underdog picks

In [9]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

underdogPairs = calculate2LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10)


underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'PROB 1', 'PROB 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']]
underdogPairs.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogPairs.csv', index=False)
underdogPairs

Pre-computing predictions for 61 players...
Processing 55 players...
Generated 1374 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,PROB 1,PROB 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
839,Jerami Grant,Miles McBride,22.5,8.5,17.10,14.76,0.847,0.834,under,over,1,10.77,0.538,Med,High
539,Alperen Sengun,Karl-Anthony Towns,23.5,21.5,28.14,26.94,0.762,0.790,over,over,1,7.70,0.385,High,High
122,Dillon Brooks,Keyonte George,18.5,18.5,22.80,23.11,0.743,0.750,over,over,1,6.38,0.319,High,High
898,Buddy Hield,Jordan Clarkson,7.5,9.5,10.83,13.22,0.724,0.728,over,over,0,5.51,0.275,Med,High
1113,Lauri Markkanen,Mikal Bridges,24.5,15.5,28.53,18.74,0.720,0.696,over,over,0,4.73,0.236,High,High
1072,Shai Gilgeous-Alexander,Tristan da Silva,31.5,12.5,28.10,15.69,0.719,0.695,under,over,0,4.69,0.234,Med,High
803,Draymond Green,Jalen Suggs,8.5,13.5,11.38,16.49,0.688,0.695,over,over,0,4.06,0.203,Med,Med
476,Nikola Jokić,Ryan Rollins,28.5,21.5,30.83,18.47,0.653,0.682,over,under,0,3.11,0.155,Med,High
1159,Ace Bailey,Goga Bitadze,10.5,5.5,12.53,7.07,0.640,0.648,over,over,0,2.20,0.110,Med,Low
44,Devin Booker,Cade Cunningham,28.5,28.5,30.64,26.25,0.623,0.632,over,under,0,1.58,0.079,High,High


### Prizepicks picks

In [10]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

prizepicksPairs = calculate2LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=15)


pairsPrizepicks = prizepicksPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'PROB 1', 'PROB 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
pairsPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksPairs.csv', index=False)
prizepicksPairs

Pre-computing predictions for 67 players...
Processing 61 players...
Generated 1688 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,MODEL SIDE 1,MODEL SIDE 2,PROB 1,PROB 2,PROB BOTH,EDGE 1,EDGE 2,COMBINED EDGE,EV$,KELLY FULL,RECOMMENDATION,SIGMA 1,SIGMA 2,SIGMA FLAG 1,SIGMA FLAG 2,CI 1,CI 2,CORRELATION,SAME_GAME,EXPECTED ROI
757,Alperen Sengun,Jerami Grant,22.5,22.5,28.14,17.10,over,under,0.807,0.847,0.6694,0.229,0.269,0.349,10.08,0.504,1,6.51,5.28,High,Med,"(15.4, 40.9)","(6.8, 27.5)",0.05,0,100.8
1466,Keyonte George,Karl-Anthony Towns,17.5,21.5,23.11,26.94,over,over,0.795,0.790,0.6152,0.216,0.212,0.294,8.46,0.423,1,6.82,6.74,High,High,"(9.7, 36.5)","(13.7, 40.2)",0.05,0,84.6
846,Aaron Gordon,Lauri Markkanen,16.5,23.5,21.33,28.53,over,over,0.785,0.766,0.5898,0.207,0.188,0.268,7.69,0.385,1,6.12,6.92,High,High,"(9.3, 33.3)","(15.0, 42.1)",0.05,0,76.9
212,Dillon Brooks,Jordan Clarkson,18.5,9.5,22.80,13.22,over,over,0.743,0.728,0.5302,0.165,0.150,0.207,5.90,0.295,0,6.60,6.12,High,High,"(9.9, 35.7)","(1.2, 25.2)",0.05,0,59.0
1355,Buddy Hield,Brandon Miller,7.5,15.0,10.83,12.34,over,under,0.724,0.725,0.5144,0.146,0.147,0.191,5.43,0.272,0,5.60,4.45,Med,Low,"(0.0, 21.8)","(3.6, 21.1)",0.05,0,54.3
135,Julius Randle,Shai Gilgeous-Alexander,22.5,31.5,26.17,28.10,over,under,0.707,0.719,0.4978,0.129,0.140,0.174,4.94,0.247,0,6.74,5.87,High,Med,"(13.0, 39.4)","(16.6, 39.6)",0.05,0,49.4
1307,Draymond Green,Mikal Bridges,8.5,15.5,11.38,18.74,over,over,0.688,0.696,0.4694,0.110,0.118,0.145,4.08,0.204,0,5.86,6.33,Med,High,"(0.0, 22.9)","(6.3, 31.2)",0.05,0,40.8
946,Cameron Johnson,Jalen Suggs,11.5,13.5,9.12,16.49,under,over,0.669,0.695,0.4556,0.091,0.117,0.131,3.67,0.183,0,5.44,5.87,Med,Med,"(0.0, 19.8)","(5.0, 28.0)",0.05,0,36.7
671,Nikola Jokić,Stephen Curry,28.5,28.5,30.83,25.76,over,under,0.653,0.663,0.4243,0.075,0.085,0.099,2.73,0.136,0,5.92,6.53,Med,High,"(19.2, 42.4)","(13.0, 38.5)",0.05,0,27.3
1544,Ace Bailey,Goga Bitadze,10.5,5.5,12.53,7.07,over,over,0.640,0.648,0.4067,0.062,0.070,0.081,2.20,0.110,0,5.65,4.14,Med,Low,"(1.5, 23.6)","(0.0, 15.2)",0.05,0,22.0


## 3 leg parlay

### Underdog picks

In [11]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

underdogTrios = calculate3LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10)

underdogTrios = underdogTrios[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'PROB 1', 'PROB 2', 'PROB 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
underdogTrios.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogTrios.csv', index=False)
underdogTrios.head()

Pre-computing predictions for 61 players...
Processing 55 players...
Generated 25615 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,PROB 1,PROB 2,PROB 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
19319,Jerami Grant,Karl-Anthony Towns,Miles McBride,22.5,21.5,8.5,17.10,26.94,14.76,0.847,0.790,0.834,under,over,over,1,20.14,0.403,Med,High,High
3141,Dillon Brooks,Alperen Sengun,Keyonte George,18.5,23.5,18.5,22.80,28.14,23.11,0.743,0.762,0.750,over,over,over,1,12.93,0.259,High,High,High
20216,Buddy Hield,Lauri Markkanen,Jordan Clarkson,7.5,24.5,9.5,10.83,28.53,13.22,0.724,0.720,0.728,over,over,over,0,10.50,0.210,Med,High,High
22929,Shai Gilgeous-Alexander,Jalen Suggs,Tristan da Silva,31.5,13.5,12.5,28.10,16.49,15.69,0.719,0.695,0.695,under,over,over,0,8.75,0.175,Med,Med,High
18835,Draymond Green,Mikal Bridges,Ryan Rollins,8.5,15.5,21.5,11.38,18.74,18.47,0.688,0.696,0.682,over,over,under,0,7.65,0.153,Med,High,High


### Prizepicks picks

In [12]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

triosPrizepicks = calculate3LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=15)


triosPrizepicks = triosPrizepicks[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'PROB 1', 'PROB 2', 'PROB 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
triosPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

Pre-computing predictions for 67 players...
Processing 61 players...
Generated 35069 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,PROB 1,PROB 2,PROB 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
20820,Alperen Sengun,Jerami Grant,Keyonte George,22.5,22.5,17.5,28.14,17.10,23.11,0.807,0.847,0.795,over,under,over,1,19.31,0.386,High,Med,High
22910,Aaron Gordon,Lauri Markkanen,Karl-Anthony Towns,16.5,23.5,21.5,21.33,28.53,26.94,0.785,0.766,0.790,over,over,over,1,15.68,0.314,High,High,High
6471,Dillon Brooks,Brandon Miller,Jordan Clarkson,18.5,15.0,9.5,22.80,12.34,13.22,0.743,0.725,0.728,over,under,over,0,11.18,0.224,High,Low,High
4563,Julius Randle,Buddy Hield,Shai Gilgeous-Alexander,22.5,7.5,31.5,26.17,10.83,28.10,0.707,0.724,0.719,over,over,under,0,9.86,0.197,High,Med,Med
24442,Cameron Johnson,Draymond Green,Mikal Bridges,11.5,8.5,15.5,9.12,11.38,18.74,0.669,0.688,0.696,under,over,over,0,7.30,0.146,Med,Med,High


In [13]:
# playerScoring('Trey Murphy III', s26, current_date, teamStarPlayer, projectedStartingFive)
# playerContext('Trey Murphy III', s26, current_date, projectedStartingFive, mainStartingFive, teamStarPlayer)